# 🎯 Basic Multimodal Content Processing

This notebook demonstrates how to build an AI agent that processes multiple content types using [Strands Agents](https://strandsagents.com/) with [Amazon Bedrock](https://aws.amazon.com/bedrock/). You'll create an agent that analyzes images and documents with the SDK's built-in `read` tool.

## What You'll Learn

- Process images and documents using the vended `read` tool (`strands.vended_tools.read`)
- Combine multiple content types in a single conversation
- Use Claude Haiku 5.5 for multimodal understanding

> The `image_reader` and `file_read` tools from `strands_tools` were removed in `strands-agents-tools` 0.9.0. The SDK-vended `read` tool returns images (png/jpg/gif/webp) and documents (pdf/docx/xlsx/...) as media blocks the model can look at directly. See the [migration guide](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/community-tools-package/).

## Prerequisites

- AWS account with [Amazon Bedrock access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
- Claude Haiku 5.5 enabled in Bedrock (`global.anthropic.claude-haiku-5-5` inference profile; check the [model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html))
- AWS CLI configured with appropriate permissions


In [ ]:
import os
import boto3
from strands.models import BedrockModel
from strands import Agent
from strands.vended_tools import read
from strands.tools import tool

# The vended `read` tool needs absolute paths
DATA = os.path.abspath("data-sample")


## 🤖 Agent Configuration


In [ ]:
# Multimodal system prompt
MULTIMODAL_SYSTEM_PROMPT = f""" You are a helpful assistant
that can process documents and images with the `read` tool.
Analyze their contents and provide relevant information.
Files live in {DATA}; always pass absolute paths to the tool.
"""
session = boto3.Session(region_name='us-east-1')

bedrock_model = BedrockModel(
    model_id="global.anthropic.claude-haiku-5-5",
    boto_session=session,
    streaming=False
)

multimodal_agent = Agent(
    system_prompt=MULTIMODAL_SYSTEM_PROMPT,
    tools=[read],
    model=bedrock_model
)


## 🎯 Usage Examples



In [ ]:
# Example 1: Image analysis
print("=== 📸 IMAGE ANALYSIS ===")
image_result = multimodal_agent(f"Analyze the image {DATA}/diagram.jpg in detail and describe everything you observe")
print(image_result)
print("\n" + "="*80 + "\n")


In [ ]:
# Claude Haiku 5.5 uses adaptive thinking, so the first block can be `reasoningContent`; collect only the text blocks
"\n".join(block["text"] for block in image_result.message["content"] if "text" in block)

In [ ]:
# Example 2: Document analysis (if you have a PDF document)
print("=== 📄 DOCUMENT ANALYSIS ===")
doc_result = multimodal_agent(f"Summarize as json the content of the document {DATA}/Welcome-Strands-Agents-SDK.pdf")
print(doc_result)


In [ ]:
# Claude Haiku 5.5 uses adaptive thinking, so the first block can be `reasoningContent`; collect only the text blocks
"\n".join(block["text"] for block in doc_result.message["content"] if "text" in block)